# Parte 2a: Modelo A (CNN custom)

> **Notebook auxiliar.** Se usa para trabajar en paralelo. Su contenido se integra en `2-train.ipynb`, que es la que se entrega.
> La lógica va en `src/lib/` (`lib.models`, `lib.training`); acá solo se llaman funciones. Al terminar, guarda los embeddings con `save_embeddings` (contrato en `src/lib/evaluation/embedding_store.py`).

**Colab:** se entrena en Google Colab (GPU T4). La primera celda clona el repo, baja las caras alineadas de Drive y deja los checkpoints en Drive, así una sesión cortada se retoma desde la última época. En local la celda solo usa el repo actual.

In [ ]:
# 0. Entorno: Colab (repo clonado + datos de Drive) o local
import sys
from pathlib import Path

EXPERIMENT = "exp2_softmax_adamw"  # cada corrida guarda sus checkpoints en su propia carpeta
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/valentinlujambio/tuia-face-recognition-app.git"
    BRANCH = "feature/tp-reconocimiento-facial"
    ROOT = Path("/content/tuia-face-recognition-app")
    DRIVE_DIR = Path("/content/drive/MyDrive/tp1")
    CKPT_DIR = DRIVE_DIR / "checkpoints" / EXPERIMENT

    if ROOT.exists():
        !git -C {ROOT} pull --ff-only
    else:
        !git clone --depth 1 --branch {BRANCH} {REPO_URL} {ROOT}
    %cd {ROOT}
    !pip install -q albumentations==2.0.8 onnx onnxruntime

    if not (ROOT / "data/processed/aligned").exists():
        archive = next(DRIVE_DIR.glob("aligned.*"))
        !mkdir -p data/processed
        if archive.suffix == ".rar":
            !apt-get -qq install -y unrar > /dev/null
            !unrar x -o+ -inul "{archive}" data/processed/
        else:
            !unzip -q -o "{archive}" -d data/processed/
else:
    ROOT = Path.cwd()
    CKPT_DIR = ROOT / "results" / "custom_cnn" / "checkpoints" / EXPERIMENT

sys.path.insert(0, str(ROOT / "src"))
print("ROOT:", ROOT, "| checkpoints:", CKPT_DIR)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from lib.dataset.augmentation import build_train_augmentation
from lib.evaluation import EmbeddingSet, results_path, save_embeddings
from lib.models import CustomCNNConfig, CustomFaceCNN, SoftmaxHead, count_parameters
from lib.training import CheckpointStore, EarlyStopping, FaceDataset, LabelEncoder, Trainer, load_faces, set_seed
from lib.vision import InputNormalization

CONFIG = json.loads((ROOT / "splits" / "config.json").read_text(encoding="utf-8"))
ALIGNED_ROOT = ROOT / CONFIG["aligned_root"]
SEED = CONFIG["seed"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(SEED)

splits = pd.read_csv(ROOT / "splits" / "splits.csv")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
splits["split"].value_counts()

## 1. Datos
- `train`: entrenamiento (80 identidades). `val_closed`: mismas identidades, para las curvas de loss/accuracy.
- `val_open`: identidades **no vistas**; su AUC de verificación elige el mejor checkpoint y dispara el early stopping, porque la app reconoce personas registradas después del entrenamiento.
- Augmentation de la Parte 1 solo en train; mismo preprocesamiento que la app (`InputNormalization`: RGB y `(x - 127.5) / 127.5`).
- Las ~13k caras de 112×112 entran en RAM (~0,5 GB en uint8), así el DataLoader no lee disco.

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2

rows = {name: splits[splits["split"] == name].reset_index(drop=True) for name in ("train", "val_closed", "val_open")}
encoder = LabelEncoder(rows["train"]["identity"])
open_encoder = LabelEncoder(rows["val_open"]["identity"])  # solo para saber qué pares son la misma persona
faces = {name: load_faces(ALIGNED_ROOT, df["aligned_path"]) for name, df in rows.items()}

normalization = InputNormalization()
datasets = {
    "train": FaceDataset(faces["train"], encoder.encode(rows["train"]["identity"]), normalization,
                         augment=build_train_augmentation()),
    "val_closed": FaceDataset(faces["val_closed"], encoder.encode(rows["val_closed"]["identity"]), normalization),
    "val_open": FaceDataset(faces["val_open"], open_encoder.encode(rows["val_open"]["identity"]), normalization),
}
loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                     persistent_workers=NUM_WORKERS > 0)
loaders = {
    "train": DataLoader(datasets["train"], shuffle=True, drop_last=True, **loader_kwargs),
    "val_closed": DataLoader(datasets["val_closed"], shuffle=False, **loader_kwargs),
    "val_open": DataLoader(datasets["val_open"], shuffle=False, **loader_kwargs),
}
pd.DataFrame({name: {"caras": len(ds), "identidades": len(np.unique(ds.labels))} for name, ds in datasets.items()}).T

## 2. Arquitectura
ResNet chica (`src/lib/models/custom_cnn.py`, importable desde el backend):

| Bloque | Salida | Detalle |
|---|---|---|
| Stem | 32×112×112 | conv 3×3 + BN + PReLU |
| Etapa 1–4 | 32×56², 64×28², 128×14², 256×7² | 2 bloques residuales por etapa, el primero con stride 2 |
| Embedding | 128 | BN → Dropout → global average pooling → FC → BN |

- **Profundidad:** 4 etapas (sugerencia del enunciado, 4–6 bloques) llevan 112 → 7; los atajos residuales permiten 8 bloques sin problemas de gradiente.
- **BatchNorm** tras cada conv: estabiliza el entrenamiento desde cero y tolera lr más altos.
- **Dropout** antes del FC: regulariza con solo ~110 imágenes por identidad.
- **Global pooling:** 256 parámetros de entrada al FC en vez de 7·7·256 = 12.544 (FC de 33k vs 1,6M parámetros); con pocos datos reduce el sobreajuste.
- **Embedding de 128-d** sin normalizar; la normalización L2 la hace quien lo consume (igual que ArcFace en la app).
- **Cabeza** (`SoftmaxHead`, `src/lib/models/heads.py`): FC 128 → 80 identidades + cross-entropy; solo para entrenar, no se exporta.

In [ ]:
model_config = CustomCNNConfig(channels=(32, 64, 128, 256), blocks_per_stage=2, embedding_dim=128, dropout=0.2)
model = CustomFaceCNN(model_config)
head = SoftmaxHead(model_config.embedding_dim, num_classes=len(encoder))

print(model)
print(f"Parámetros extractor: {count_parameters(model):,} | cabeza: {count_parameters(head):,}")

## 3. Hiperparámetros

| | Valor | Por qué |
|---|---|---|
| Pérdida | cross-entropy (softmax) | enfoque 1 del enunciado: clasificación + embedding |
| Optimizador | AdamW | el que usa la cátedra (clases 2 y 4); robusto al lr y converge rápido desde cero |
| Learning rate | 1e-3 con warmup lineal 2 épocas + coseno | warmup evita pasos grandes con estadísticas de Adam todavía inestables |
| Weight decay | 1e-4 | decoupled weight decay (AdamW), mismo valor que la cátedra |
| Batch size | 128 | entra holgado en una T4 con AMP |
| Épocas | 60 máx., early stopping con paciencia 10 | sobre AUC de verificación en val_open |
| AMP | float16 en GPU | ~2x más rápido en T4 |

In [ ]:
EPOCHS = 60
LR = 1e-3
WARMUP_EPOCHS = 2
WEIGHT_DECAY = 1e-4
PATIENCE = 10

params = list(model.parameters()) + list(head.parameters())
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS),
        torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS),
    ],
    milestones=[WARMUP_EPOCHS],
)
trainer = Trainer(model, head, optimizer, scheduler, DEVICE, amp=True)
store = CheckpointStore(CKPT_DIR)
early_stopping = EarlyStopping(patience=PATIENCE)

## 4. Entrenamiento
Cada época guarda `last.pt` (estado completo) e `history.json` en Drive; `best.pt` cuando mejora el AUC de val_open (marcado con `*`).
Si Colab corta la sesión, volver a correr la notebook reanuda desde la última época. Para empezar de cero: `resume=False` o borrar la carpeta de checkpoints.

In [ ]:
history = trainer.fit(
    loaders["train"], loaders["val_closed"], loaders["val_open"],
    epochs=EPOCHS, early_stopping=early_stopping, store=store, resume=True,
)

## 5. Curvas de entrenamiento
Loss y accuracy de train vs. validation (val_closed) y AUC/EER de verificación en val_open.

In [ ]:
hist = pd.DataFrame(json.loads(store.history_path.read_text(encoding="utf-8")))
best_epoch = int(hist.loc[hist["val_open_auc"].idxmax(), "epoch"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (metric, title) in zip(axes[:2], [("loss", "Loss"), ("acc", "Accuracy")]):
    ax.plot(hist["epoch"], hist[f"train_{metric}"], label="train")
    ax.plot(hist["epoch"], hist[f"val_{metric}"], label="val_closed")
    ax.set_title(title)
axes[2].plot(hist["epoch"], hist["val_open_auc"], label="AUC")
axes[2].plot(hist["epoch"], hist["val_open_eer"], label="EER")
axes[2].set_title("Verificación en val_open")
for ax in axes:
    ax.axvline(best_epoch, color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("época")
    ax.legend()
plt.tight_layout()
plt.show()
hist.loc[hist["epoch"] == best_epoch]

## 6. Experimentos

Mejor época de cada corrida (checkpoints en `MyDrive/tp1/checkpoints/<experimento>`).

| # | Experimento | Cambio | Mejor época | Val loss | Val acc | AUC val_open | EER val_open | Conclusión |
|---|---|---|---|---|---|---|---|---|
| 1 | `exp1_softmax_sgd` | Baseline: softmax + SGD (lr 0,1, momentum 0,9, wd 5e-4), 4 etapas × 2 bloques, emb 128, dropout 0,2 | 41 / 51 | 0,369 | 0,909 | 0,893 | 0,188 | Train acc 0,99 vs val 0,91: sobreajuste leve. AUC val_open estancado en ~0,88–0,89 desde la época 30 mientras val_closed sigue mejorando: softmax separa las 80 identidades de train pero no generaliza tanto a personas nuevas. |
| 2 | `exp2_softmax_adamw` | Igual que 1 con AdamW (lr 1e-3, wd 1e-4), como en la cátedra | | | | | | |

## 7. Exportación
`torch.onnx.export` a `models/custom_cnn.onnx` (no se versiona; subir a Drive/HF y poner el link en el README).

In [ ]:
# TODO (segmento 3): cargar best.pt y exportar a ONNX

## 8. Embeddings para la evaluación
Se calculan sobre todos los splits salvo `train` y se guardan con el contrato común.

In [ ]:
# TODO (segmento 3): embeddings con el .onnx exportado y save_embeddings